# 06 · Catalog + Process: Sentinel-2 NDVI time series

**Objective:** Discover acquisitions, request narrow timestamp windows, and export compact per-acquisition summaries without retaining all images.

**Route:** Live API / CDSE OAuth client · **Audience:** researcher / developer · **Duration:** 15–30 minutes after setup.

[Notebook index](README.md) · [Environment setup](../docs/setup/README.md) · [Accounts and API keys](../docs/setup/accounts.md) · [Data quality](../docs/reference/data-quality.md) · [Guide home](../README.md)

**Verification:** authored from the original workflow and official specifications; live access and your input files have **not been tested in this guide**. All distributed code cells have empty outputs. The release verification report records the separate offline checks. Run from a fresh kernel in numbered cell order.

**Expected result:** a dated summary table, a time-series plot and provenance. Empty searches or missing permissions are possible; they are not evidence of a scientific result.

[Service overview and access route](../docs/services/cdse-sentinel-hub.md)


## 1. Load the local environment

Install the documented environment before opening this notebook. Paths in `.env` are relative to the repository root. Authentication is kept outside committed notebooks.


In [ ]:
from pathlib import Path
import sys
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "examples" / "common.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter inside DestinE-partner-guide.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env", override=False)
from examples.common import required_env, input_path, public_url, record_provenance


## 2. Search a bounded date interval

Catalogue filtering is separate from raster processing. The first page is capped at 100 items and this tutorial processes at most five unique timestamps. A full study must implement pagination and verify acquisition completeness. Requests use ±60 seconds around each catalogue timestamp, rather than a whole-day mosaic.


In [ ]:
from examples.sentinel import access_token, catalog, process_payload, process, narrow_window
from examples.raster import ndvi
COLLECTION = "sentinel-2-l2a"
bbox = [24.55, 60.15, 24.57, 60.17]
start, end = "2024-08-01T00:00:00Z", "2024-08-31T23:59:59Z"
orbit = "ASCENDING"


In [ ]:
# Read the request and account prerequisites before changing this switch.
RUN_LIVE = False
if not RUN_LIVE:
    raise RuntimeError("Live retrieval is disabled. Complete account setup, review quotas, then set RUN_LIVE=True.")


In [ ]:
token = access_token()
features, more_pages = catalog(token, COLLECTION, bbox, start, end, max_cloud=20, orbit=orbit)
print({"items_on_first_page": len(features), "more_pages": more_pages})
if not features:
    raise ValueError("No catalogue items match these filters.")
catalog_table = pd.DataFrame([{ "id": f["id"], "time": f["properties"]["datetime"], "cloud": f["properties"].get("eo:cloud_cover"), "orbit": f["properties"].get("sat:orbit_state"), "relative_orbit": f["properties"].get("sat:relative_orbit")} for f in features])
catalog_table.head(10)


## 3. Reduce each masked optical acquisition


The Process request retains only SCL vegetation, bare soil and water. Record the valid count alongside every mean; a changing mask can change the sampled population.


In [ ]:
rows = []
seen = set()
for feature in sorted(features, key=lambda item: item["properties"]["datetime"]):
    timestamp = feature["properties"]["datetime"]
    if timestamp in seen:
        continue  # tile records at the same time can describe one acquisition
    seen.add(timestamp)
    interval_start, interval_end = narrow_window(timestamp, seconds=60)
    payload = process_payload(COLLECTION, bbox, interval_start, interval_end, orbit=orbit, max_cloud=20)
    image, grid = process(token, payload)
    red, nir, mask, scl = image
    result = ndvi(red, nir, mask > 0)
    valid = np.isfinite(result)
    measurements = {"ndvi_mean": float(np.nanmean(result)) if valid.any() else np.nan}
    rows.append({"catalog_item": feature["id"], "acquisition": timestamp, "request_from": interval_start, "request_to": interval_end, "valid_count": int(valid.sum()), "relative_orbit": feature["properties"].get("sat:relative_orbit"), **measurements})
    if len(rows) >= 5:
        break  # teaching cap; this is not a complete monthly series
table = pd.DataFrame(rows)
if table.empty:
    raise ValueError("No acquisitions processed. Check collection, dates, geometry and filters.")
table


## 4. Plot and export the compact table


In [ ]:
table["acquisition"] = pd.to_datetime(table["acquisition"], utc=True)
table.plot(x="acquisition", y=['ndvi_mean'], marker="o", figsize=(9, 4))
plt.ylabel('Mean NDVI')
plt.title("Sentinel-2 NDVI — up to five sample acquisitions")
plt.show()
(ROOT / "outputs").mkdir(exist_ok=True)
table.to_csv(ROOT / "outputs" / "sentinel-2-time-series.csv", index=False)
record_provenance("sentinel-2-time-series.json", collection=COLLECTION, bbox_wgs84=bbox, search_interval=[start, end], catalog_ids=table["catalog_item"].tolist(), first_page_only=True, more_pages=more_pages, maximum_acquisitions=5, process_window_seconds=120, orbit_direction=orbit if COLLECTION == "sentinel-1-grd" else None)


## Interpret and verify

Acquisition-level means use only valid pixels; differing cloud masks can bias comparisons. Match site polygons, seasonal sampling and observation coverage before detecting trends. The capped first page/five-acquisition demonstration is not a complete August time series.

Check CRS, dates, masks, units, spatial extent, and valid sample count before comparing results. Keep product IDs and request parameters with the exported output. Do not treat synthetic/offline checks as verification of a remote service.


## Official references

- [CDSE Process API](https://documentation.dataspace.copernicus.eu/APIs/SentinelHub/Process.html)
- [CDSE authentication](https://documentation.dataspace.copernicus.eu/APIs/SentinelHub/Overview/Authentication.html)
- [CDSE Catalog API](https://documentation.dataspace.copernicus.eu/APIs/SentinelHub/Catalog.html)

[Return to notebook index](README.md) · [Continue through Start here](../docs/start-here.md) · [Guide home](../README.md)
